> ### **Target Clean**

In [0]:
BRONZE_PATH = "/Volumes/workspace_team3/default/bronze_layer/"
SILVER_PATH = "/Volumes/workspace_team3/default/silver_layer/"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import pyspark.sql.functions as F

In [0]:
target_schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("Target", StringType(), True),
    StructField("TargetMonth", StringType(), True)
])

In [0]:
df_target_raw = (spark.read
                 .format("csv")
                 .schema(target_schema)
                 .option("header", "true")
                 .option("sep", "\t")
                 .option("ignoreLeadingWhiteSpace", "true")
                 .option("ignoreTrailingWhiteSpace", "true")
                 .option("quote", '"')
                 .option("escape", '"')
                 .load(BRONZE_PATH + "Targets.csv")
                )

In [0]:
display(df_target_raw)

In [0]:
df_target_silver = (df_target_raw
                    
    #Μετατροπή του EmployeeID από String σε Integer
    #.withColumn("EmployeeID", F.col("EmployeeID").cast(IntegerType()))

    #Καθαρισμός Νομίσματος: Αφαιρούμε το $ και το κόμμα, και το κάνουμε Double (Δεκαδικό)
    .withColumn("Target", F.regexp_replace(F.col("Target"), r'[\$,]', '').cast(DoubleType()))

    # Καθαρισμός Ημερομηνίας (π.χ. 'Friday, December 1, 2017' -> 2017-12-01)
    # Απομονώνουμε μόνο το "December 1, 2017" αγνοώντας την ημέρα για να μην κολλήσει το Spark 3.0+
    .withColumn("CleanDate", F.substring_index(F.col("TargetMonth"), ", ", -2))
    .withColumn("TargetMonth", F.to_date(F.col("CleanDate"), "MMMM d, yyyy"))
    .drop("CleanDate") # Πετάμε την προσωρινή στήλη

    # Αφαίρεση διπλοτύπων
    .dropDuplicates()
)

In [0]:
display(df_target_silver)

## **Transform to Parquet**

In [0]:
df_target_silver.write.mode("overwrite").parquet("/Volumes/workspace_team3/default/silver_layer/targets_clean.parquet")